# Fine tuning a BETO
Este ejercicio consiste en realizar fine tunning al modelo de BETO para una tarea de NER (Named Entity Recognition) sobre un corpus basado en sentencias médicas. Se utiliza las librerías de Hugging Face (`transformers`, `datasets`, `evaluate`, etc.) para esta tarea.

# Instalación de dependencias

In [1]:
%pip install datasets evaluate transformers[sentencepiece] accelerate seqeval
%pip install huggingface_hub<1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 927.8 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 486.3 kB/s eta 0:00:00a 0:00:01
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16161 sha256=d9dff051cdbcd2b2171d55100e4d1384c718f65907cac01a10eddce360920ce6
  Stored in directory: /root/.cache/pip/wheels/1a/67/4a/ad4082dd7dfc30f2abfe4d80a2ed5926a506eb8a972b4767fa
Successfully built seqeval
Note: you may need to restart the kernel to use updated packages.
/bin/bash: line 1: 1.0: No such file or directory
Note: you may need to restart the kernel to use updated packages.


Se desactivan las advertencias

In [5]:
import warnings

warnings.filterwarnings("ignore")

# Inicio de sesión en Hugging Face

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
TOKEN_HUGGINGFACE = user_secrets.get_secret("API_HUGGINGFACE")
login(TOKEN_HUGGINGFACE)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


# Carga de datos
El cojunto de datos se carga desde un script de Python, este consta de 3 particiones: entrenamiento, prueba y validación. Este solo contiene las sentencias y las NER tags.

In [3]:
from datasets import load_dataset

RUTA = "/kaggle/input/biobert/Biobert/Biobert_json.py"
raw_datasets = load_dataset(RUTA, trust_remote_code=True)
raw_datasets

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentencia', 'tag'],
        num_rows: 9788
    })
    validation: Dataset({
        features: ['sentencia', 'tag'],
        num_rows: 2758
    })
    test: Dataset({
        features: ['sentencia', 'tag'],
        num_rows: 2496
    })
})

Cada instancia es una frase tokenizada por subpalabras mediante WordPiece, a su vez cada token tiene asignada su NER TAG respectiva.

In [4]:
print(f"Tokens: {raw_datasets['train'][0]['sentencia']}")
print(f"NER TAGS: {raw_datasets['train'][0]['tag']}")

Tokens: ['Abuela', 'materna', 'con', 'cancer', 'de', 'mama', 'a', 'los', '70', 'años', '.']
NER TAGS: [4, 19, 29, 0, 16, 16, 29, 29, 10, 8, 29]


Ahora observemos el texto correspondiente a cada una de las etiquetas que vimos previamente. Notese que casi todas las etiquetas están relacionadas con conceptos médicos, algunas hacen referencia a tratamientos, personas, etc. Resalta la etiqueta "O" que corresponde a aquellos tokens que no son relevantes a la temática como lo pueden ser signos de puntuación o stopwords.

In [15]:
ner_feature = raw_datasets["train"].features["tag"]
label_names = ner_feature.feature.names
print(label_names)

['B_CANCER_CONCEPT', 'B_CHEMOTHERAPY', 'B_DATE', 'B_DRUG', 'B_FAMILY', 'B_FREQ', 'B_IMPLICIT_DATE', 'B_INTERVAL', 'B_METRIC', 'B_OCURRENCE_EVENT', 'B_QUANTITY', 'B_RADIOTHERAPY', 'B_SMOKER_STATUS', 'B_STAGE', 'B_SURGERY', 'B_TNM', 'I_CANCER_CONCEPT', 'I_DATE', 'I_DRUG', 'I_FAMILY', 'I_FREQ', 'I_IMPLICIT_DATE', 'I_INTERVAL', 'I_METRIC', 'I_OCURRENCE_EVENT', 'I_SMOKER_STATUS', 'I_STAGE', 'I_SURGERY', 'I_TNM', 'O']


# Proceso de fine tunning

Utilizaremos un modelo basado en BERT para nuestra tarea de NER, sin embargo, ya existe una variante de BERT denominada BETO que fue entrenada utilizando un corpus en español. Existen muchas variantes BETO, pero en este caso utilizaremos `BETO-uncased` cuya particularidad es que no distingue mayúsculas y minúsculas lo que es beneficioso para nuestro caso.

## Cargando el tokenizador

In [8]:
from transformers import AutoTokenizer

MODELO_BASE = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE, add_prefix_space=True)

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Probemos el tokenizador con una instancia, observemos que este añade un token especial al inicio y al final de la oración, esto es una característica de BERT que ayuda a brindarle contexto al modelo.

In [9]:
inputs = tokenizer(raw_datasets["train"][0]["sentencia"], is_split_into_words=True)
print(inputs.tokens())

['[CLS]', 'abuela', 'materna', 'con', 'cancer', 'de', 'mama', 'a', 'los', '7', '##0', 'años', '.', '[SEP]']


Tokenizamos todas las sentencias del conjunto de datos (todas las particiones) y alineamos las etiquetas.

In [10]:
def tokenize_and_align_labels(instancias):
    # Se tokeniza la sentencia
    tokenized_inputs = tokenizer(instancias["sentencia"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(instancias["tag"]):
        # Se obtiene el id de cada palabra dentro del vocabulario
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Se asigna el valor de -100 a las palabras que tienen tokens especiales. Se ignoran en la función 
            # de error
            if word_idx is None:
                label_ids.append(-100)
            # Se asigna la etiqueta adecuada para el 1° token de cada palabra
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Al resto de tokens de una misma palabra se le asigna la misma etiqueta
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    # Almacenamos las etiquetas asignadas a las instancias del lote
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [11]:
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/9788 [00:00<?, ? examples/s]

Map:   0%|          | 0/2758 [00:00<?, ? examples/s]

Map:   0%|          | 0/2496 [00:00<?, ? examples/s]

## Añadiendo el dataloader

La librerías `transformers` está basada en Pytorch, por lo que al igual que en esta se define un Dataset y luego para separar las instancias por lotes y pasarlas al modelo se requiere de un DataLoader. La clase `DataCollator` cumple esta función, en este caso, pasa instancias individuales al modelo y las tokeniza.

In [12]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## Ver las métricas.

In [13]:
from numpy import argmax
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    # Los logits ya tienen aplicada la función softmax, solo resta seleccionar la clase con mayor probabilidad
    logits, etiquetas = eval_preds
    preds = argmax(logits, axis=-1)

    # Se retiran los tokens que indican el inicio y fin de la sentencia y convertimos el id numérico de cada
    # etiqueta al texto que le corresponde
    y = [[label_names[l] for l in etiqueta if l != -100] for etiqueta in etiquetas]

    # Repetimos el mismo proceso para las etiquetas predecidas por el modelo en cada token
    y_gorro = [
        [label_names[p] for (p, l) in zip(pred, etiqueta) if l != -100]
        for pred, etiqueta in zip(preds, etiquetas)
    ]

    all_metrics = metric.compute(predictions=y_gorro, references=y)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

Se crea un mapeo de cada etiqueta con su id respectivo dentro del conjunto de datos y viceversa.

In [16]:
label_names = ner_feature.feature.names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

## Instanciando el modelo

Se descarga e instancia el modelo de BETO para una tarea de clasificación de tokens. Esto añade una capa de clasificación al final que será entrenada con nuestro conjunto de datos y a su vez congelará (evitará que las otras capas del modelo "pierdan sus conocimientos").

In [20]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODELO_BASE,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=False,
    num_labels=len(label_names)
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Entrenamiento del modelo
### Definición de hiperparámetros

Se definen algunos hiperparámetros a utilizar para el entrenamiento como la tasa de aprendizaje y el batch size. También se configuran aspectos del early stopping como la métrica a evaluar y la estrategia de guardado del modelo. También se establece el nombre del modelo (respositorio de Hugging Face).

In [18]:
from transformers import TrainingArguments

args = TrainingArguments(
    "NER-finetuning-BETO-UNCASED-BIOBERT",
    evaluation_strategy="epoch",           # Las métricas se evaluan por cada iteración
    save_strategy="epoch",                 # Se guarda un punto de control del entrenamiento por cada iteración
    learning_rate=2e-5,                    # Tasa de aprendizaje
    num_train_epochs=10,                   # Entrenamiento del modelo por 10 iteraciones
    weight_decay=0.01,                     # Factor de regularización L2
    push_to_hub=False,                     # Para activar la publicación del modelo automaticamente se termine el entrenamiento
    per_device_train_batch_size=32,        # Batch size del conjunto de entrenamiento
    per_device_eval_batch_size=32,         # Batch size del conjunto de validación/prueba
    report_to="none",                      # Las métricas no se almacenan en ningún lado
    metric_for_best_model="eval_accuracy", # Se asigna el "accuracy" como métrica a medir en el early stopping
    load_best_model_at_end=True			   # Activando el early stopping
)

### Proceso de entrenamiento
Finalmente, se pasan los hiperparámetros desginados anteriormente, las particiones de entrenamiento y prueba, el modelo base, el tokenizador y el data collator. Además, se configura el early stopping para que detenga el entrenamiento si  accuracy no aumenta al menos un 10% durante 3 épocas. 

In [21]:
from transformers import Trainer, EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(early_stopping_patience = 3, early_stopping_threshold = 0.1)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[early_stopping]
)
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.127665,0.907150,0.953516,0.929756,0.964676
2,0.313400,0.121585,0.913439,0.964033,0.938054,0.968280
3,0.313400,0.112275,0.930504,0.966907,0.948356,0.972216
4,0.082800,0.119579,0.923438,0.966557,0.944505,0.970642


TrainOutput(global_step=1224, training_loss=0.1739546761793249, metrics={'train_runtime': 374.9787, 'train_samples_per_second': 261.028, 'train_steps_per_second': 8.16, 'total_flos': 1751831821272240.0, 'train_loss': 0.1739546761793249, 'epoch': 4.0})

## Evaluando el rendimiento del modelo con el conjunto de validación

In [24]:
preds, etiquetas, _ = trainer.predict(tokenized_datasets["validation"])

print(compute_metrics((preds, etiquetas)))

{'precision': 0.9466807516957634, 'recall': 0.9644293401302747, 'f1': 0.9554726298364243, 'accuracy': 0.9764963770322109}


# Subir el modelo a Hugging face

In [51]:
trainer.push_to_hub()

events.out.tfevents.1731104105.7b5fca91546f.30.0:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Criser2013/NER-finetuning-BETO-UNCASED-BIOBERT/commit/5f5b38a125d45f2661e5f6be8d3b67a15ca3367f', commit_message='End of training', commit_description='', oid='5f5b38a125d45f2661e5f6be8d3b67a15ca3367f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Criser2013/NER-finetuning-BETO-UNCASED-BIOBERT', endpoint='https://huggingface.co', repo_type='model', repo_id='Criser2013/NER-finetuning-BETO-UNCASED-BIOBERT'), pr_revision=None, pr_num=None)